# Training on Colab: offline early exit voice assistant

**Before you start**
1. Runtime > Change runtime type > pick a GPU (L4 or A100 if you have them with Pro, otherwise T4).
2. Upload the whole `gpu_package` folder to your Google Drive (My Drive). It must contain `gpu_training_package.zip` and the `data_files` folder.
3. Run the cells from top to bottom. Cells 1 to 5 are setup. Then each training cell is one stage.

**If Colab disconnects:** reconnect, run the setup cells (1 to 5) again, then run the training cell that was running. The `--resume` flag continues from the last saved epoch.
Checkpoints, exports, results and logs are saved straight to Drive in `gpu_package/out`, so nothing is lost when the runtime resets.

**To cut the time:** set `EXTRA = "--epochs 10"` in cell 1. Tell us what you used.

## 1. Connect Drive and settings

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/gpu_package'
OUT = DRIVE + '/out'
EXTRA = ''          # for example '--epochs 10'
import os
assert os.path.exists(DRIVE + '/gpu_training_package.zip'), 'upload the gpu_package folder to My Drive first'
os.makedirs(OUT, exist_ok=True)

## 2. Check the GPU

In [ ]:
import torch
print(torch.__version__, 'cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU, then start again'
print(torch.cuda.get_device_name(0))
!nvidia-smi

## 3. Unpack the code and the data (about 10 minutes, skips what is already done)

In [ ]:
%%bash -s "$DRIVE" "$OUT"
set -e
DRIVE=$1; OUT=$2
mkdir -p /content/proj && cd /content/proj
[ -d src ] || unzip -q -o "$DRIVE/gpu_training_package.zip"
# keep checkpoints, exports, results and logs on Drive so a disconnect cannot lose them
mkdir -p "$OUT/checkpoints" "$OUT/export" "$OUT/results" "$OUT/logs"
rm -rf checkpoints export results
ln -s "$OUT/checkpoints" checkpoints; ln -s "$OUT/export" export; ln -s "$OUT/results" results
# data goes on the fast local disk, not on Drive
mkdir -p data/raw/fsc_zenodo data/raw/speech_commands_v2
ls data/raw/fsc_zenodo/*.tar >/dev/null 2>&1 || cp "$DRIVE"/data_files/fluentspeechcommands_*.tar data/raw/fsc_zenodo/
[ -f data/raw/speech_commands_v2/validation_list.txt ] || tar -xzf "$DRIVE/data_files/speech_commands_v0.02.tar.gz" -C data/raw/speech_commands_v2
echo "checking FSC files..."
(cd data/raw/fsc_zenodo && grep fluentspeechcommands "$DRIVE/data_files/checksums.md5" | md5sum -c -)
ls data/raw/speech_commands_v2 | head -5

## 4. Install the packages

In [ ]:
!pip install -q transformers soundfile ai-edge-litert pyyaml tqdm psutil

## 5. Prepare the data (expect 84843 / 9981 / 11005 and 23132 / 3118 / 3793 clips)

In [ ]:
%cd /content/proj
!python data/prepare.py sc
!python data/prepare.py fsc

## 6. Timing test: 1 epoch of the teacher
Write down how long this takes. If it runs out of memory, lower `batch_size` under `train:` in `configs/train_encoder.yaml` (for example 16).
If it crashes for any other reason, send us the full error text.

In [ ]:
%cd /content/proj
!time python -u -m src.encoder.distill --stage teacher --epochs 1 2>&1 | tee {OUT}/logs/teacher_smoke.log

## 7. Wake word model (independent, Speech Commands only)

In [ ]:
%cd /content/proj
!python -u -m src.wakeword.train --run_name wakeword 2>&1 | tee {OUT}/logs/wakeword.log

## 8. Teacher: Whisper base fine tuned on FSC
(continues from the 1 epoch test above, that is fine)

In [ ]:
%cd /content/proj
!python -u -m src.encoder.distill --stage teacher --resume {EXTRA} 2>&1 | tee -a {OUT}/logs/teacher.log

## 9. Student: Whisper tiny, distilled from the teacher

In [ ]:
%cd /content/proj
!python -u -m src.encoder.distill --stage student --resume {EXTRA} 2>&1 | tee -a {OUT}/logs/student.log

## 10. Exit heads for lambda 1, 2, 5, 10 (distilled student)

In [ ]:
%cd /content/proj
!python -u -m src.encoder.train_exits --encoder checkpoints/student/encoder.pt --tag distill --resume {EXTRA} 2>&1 | tee -a {OUT}/logs/exits_distill.log

## 11. Naive FP32 baseline

In [ ]:
%cd /content/proj
!python -u -m src.baselines.naive {EXTRA} 2>&1 | tee -a {OUT}/logs/naive.log

## 12. Optional: the no distillation ablation
Only needed for the distillation ablation. Run both cells.

In [ ]:
%cd /content/proj
!python -u -m src.encoder.distill --stage student --no_distill --resume {EXTRA} 2>&1 | tee -a {OUT}/logs/student_nodistill.log

In [ ]:
%cd /content/proj
!python -u -m src.encoder.train_exits --encoder checkpoints/student_nodistill/encoder.pt --tag nodistill --resume {EXTRA} 2>&1 | tee -a {OUT}/logs/exits_nodistill.log

## 13. GPU tests (tell us the output either way)

In [ ]:
%cd /content/proj
!python -m pytest -m gpu -q 2>&1 | tee {OUT}/logs/gpu_tests.log

## 14. What to send back
Everything is already in `My Drive/gpu_package/out`: `checkpoints/`, `export/`, `results/` and `logs/`.
Right click the `out` folder in Drive, download it as a zip, and send it. Also tell us the epoch counts you used and roughly how long each step took.

In [ ]:
!ls -la {OUT}/checkpoints {OUT}/export {OUT}/results